# redMaPPer input check, v2 runs (12 versions), from the DataRegistry

Three setups, all b < -20 and dust corrected (E(B-V) = 0.86 x CSFD, before the refmag < 24 cut):

| Setup | Bands | Visits |
|---|---|---|
| riz2 | riz | >= 2 |
| riz3 | riz | >= 3 |
| griz2 | griz | >= 2 |

each with no star mask and with Gaia star masks of only the brighter stars (G < 12, 15, 16, same circle sizes as the
shear team's mask). Everything is read from the DataRegistry

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skyproj
import healpy as hp
import healsparse as hsp
import fitsio
import yaml
import redmapper
from dataregistry import DataRegistry

datareg = DataRegistry()
pa = hp.nside2pixarea(4096, degrees=True)   # nside 4096 pixel area, deg^2
out_dir = '/pscratch/sd/d/dericj/dp2_redmapper/rmcheck_v2'   # redmapper configs go here
os.makedirs(out_dir, exist_ok=True)

In [ ]:
def get_path(name):
    # folder (ours are external) of the newest version of a dataset
    r = datareg.query.find_datasets(['dataset.dataset_id', 'dataset.version_string', 'dataset.location_type', 'dataset.url'],
                                    [datareg.query.gen_filter('dataset.name', '==', name)],
                                    return_format='dataframe', strip_table_names=True)
    row = max(r.itertuples(), key=lambda x: [int(v) for v in x.version_string.split('.')])
    if row.location_type == 'external':
        return row.url
    return datareg.get_dataset_absolute_path(row.dataset_id)


def one_file(name, pattern):
    return glob.glob(get_path(name) + '/' + pattern)[0]

## Get the files

In [ ]:
setups = {'riz2': ('riz', 2), 'riz3': ('riz', 3), 'griz2': ('griz', 2)}
stars = ['nostar', 'star12', 'star15', 'star16']

files = {}
bands = {}
for s, (b, vis) in setups.items():
    for star in stars:
        name = f'{b}_vis{vis}_bsouth20_{star}'
        v = (s, star)
        bands[v] = list(b)
        files[v] = {'galaxies': one_file(f'dp2_redmapper_galaxies_{name}', '*master_table.fit'),
                    'geometry': one_file(f'dp2_redmapper_geometry_{name}', '*.hs'),
                    'depth': one_file(f'dp2_redmapper_depth_min_{name}', '*.hs'),
                    'specz': one_file(f'dp2_redmapper_specz_desi_dr1_red_{name}', '*.fit')}
print(len(files), 'versions, e.g.')
print(files[('griz2', 'star16')])

## Footprint

In [ ]:
area = {}
fig, axs = plt.subplots(3, 4, figsize=(30, 16))
for row, s in zip(axs, setups):
    for ax, star in zip(row, stars):
        m = hsp.HealSparseMap.read(files[(s, star)]['geometry'])
        area[(s, star)] = m[m.valid_pixels].sum() * pa
        sp = skyproj.McBrydeSkyproj(ax=ax)
        sp.draw_hspmap(m, cmap='viridis', vmin=0, vmax=1)
        sp.ax.set_title(f'{s} {star}, {area[(s, star)]:.1f} deg$^2$', pad=30)
plt.tight_layout()
plt.show()

print(pd.DataFrame({s: [area[(s, star)] for star in stars] for s in setups}, index=stars).round(1))

## Depth (shallowest band, dust corrected)

The star mask doesn't change the depth, so one map per setup.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(24, 6))
for ax, s in zip(axs, setups):
    d = hsp.HealSparseMap.read(files[(s, 'nostar')]['depth'])
    sp = skyproj.McBrydeSkyproj(ax=ax)
    sp.draw_hspmap(d['limmag'], cmap='magma', vmin=22.5, vmax=25)
    sp.ax.set_title(s, pad=30)
sp.draw_colorbar(label='10 sigma limmag (shallowest band)')
plt.show()

for v in files:
    d = hsp.HealSparseMap.read(files[v]['depth'])
    lim = d[d.valid_pixels]['limmag']
    hdr = fitsio.read_header(files[v]['depth'], ext='COV')   # redmapper reads ZP / NSIG from here
    print(f'{v[0]:6s} {v[1]:7s} median {np.median(lim):.2f}, 10-90% {np.percentile(lim, 10):.2f}-{np.percentile(lim, 90):.2f}, '
          f'ZP {hdr["ZP"]}, NSIG {hdr["NSIG"]}')

In [ ]:
mag_bins = np.linspace(14, 24.5, 106)
color_bins = np.linspace(-0.5, 2.5, 121)
refmag_hist = {}
color_hist = {}
n_galaxies = {}

for v in files:
    master = fitsio.read(files[v]['galaxies'])
    folder = os.path.dirname(files[v]['galaxies'])
    nb = len(bands[v])
    refmag_hist[v] = np.zeros(len(mag_bins) - 1)
    color_hist[v] = np.zeros((nb - 1, len(color_bins) - 1))

    n_gal = 0
    n_bad = 0
    n_wrong_ngals = 0
    for fn, ngals in zip(master['filenames'][0], master['ngals'][0]):
        g = fitsio.read(os.path.join(folder, fn.strip()), columns=['refmag', 'refmag_err', 'mag', 'mag_err'])
        ok = (np.isfinite(g['refmag']) & np.isfinite(g['refmag_err']) & np.all(np.isfinite(g['mag']), axis=1)
              & np.all(np.isfinite(g['mag_err']), axis=1) & np.all((g['mag'] > 0) & (g['mag'] < 90), axis=1)
              & np.all((g['mag_err'] > 0) & (g['mag_err'] < 90), axis=1))
        n_bad += (~ok).sum()
        n_wrong_ngals += len(g) != ngals
        n_gal += len(g)
        refmag_hist[v] += np.histogram(g['refmag'], bins=mag_bins)[0]
        for k in range(nb - 1):
            color_hist[v][k] += np.histogram(g['mag'][:, k] - g['mag'][:, k + 1], bins=color_bins)[0]

    n_galaxies[v] = n_gal
    print(f'{v[0]:6s} {v[1]:7s} {n_gal} galaxies in {len(master["filenames"][0])} files, {n_gal / master["area"][0]:.0f} per deg^2, '
          f'{n_bad} bad, {n_wrong_ngals} files where ngals is wrong, bands {bands[v]}, zeropoint {master["zeropoint"][0]}')

In [ ]:
centers = 0.5 * (mag_bins[1:] + mag_bins[:-1])
fig, axs = plt.subplots(2, 3, figsize=(20, 9), sharex=True)
for k, s in enumerate(setups):
    for star in stars:
        dens = refmag_hist[(s, star)] / area[(s, star)]
        axs[0, k].step(centers, dens, where='mid', label=star)
        with np.errstate(invalid='ignore', divide='ignore'):
            axs[1, k].step(centers, dens / (refmag_hist[(s, 'nostar')] / area[(s, 'nostar')]), where='mid', label=star)
    axs[0, k].set_yscale('log'); axs[0, k].set_title(s); axs[0, k].legend()
    axs[1, k].set_ylim(0, 1.5); axs[1, k].axhline(1, color='k', lw=0.5)
    axs[1, k].set_xlabel('refmag (cModel z, dust corrected)')
axs[0, 0].set_ylabel('galaxies per deg$^2$ per bin'); axs[1, 0].set_ylabel('density / no star mask')
plt.show()

In [ ]:
## colors, no star mask
c = 0.5 * (color_bins[1:] + color_bins[:-1])
fig, axs = plt.subplots(1, 3, figsize=(20, 4.5))
for ax, color in zip(axs, ['g - r', 'r - i', 'i - z']):
    for s in setups:
        b = bands[(s, 'nostar')]
        for k in range(len(b) - 1):
            if f'{b[k]} - {b[k + 1]}' == color:
                ax.step(c, color_hist[(s, 'nostar')][k] / area[(s, 'nostar')], where='mid', label=s)
    ax.set_xlabel(color + ' (dust corrected)'); ax.set_ylabel('per deg$^2$'); ax.legend()
plt.show()

## Red spec-z

In [ ]:
n_red = {}
fig, axs = plt.subplots(1, 3, figsize=(20, 4.5), sharey=True)
for ax, s in zip(axs, setups):
    for star in stars:
        sz = fitsio.read(files[(s, star)]['specz'])
        n_red[(s, star)] = len(sz)
        ax.hist(sz['z'], bins=np.linspace(0, 1.2, 49), histtype='step', label=f'{star} ({len(sz)})')
        if sz['z_err'].max() >= 0.001:
            print('z_err too big in', s, star)
    ax.set_title(s); ax.set_xlabel('z'); ax.legend()
axs[0].set_ylabel('red spec-z')
plt.show()

In [ ]:
def make_config(v):
    nb = len(bands[v])
    cfg = {'galfile': files[v]['galaxies'], 'specfile': files[v]['specz'],
           'maskfile': files[v]['geometry'], 'depthfile': files[v]['depth'], 'mask_mode': 3,
           'refmag': 'z', 'bands': bands[v], 'zrange': [0.1, 0.9], 'survey_mode': 0,
           'mstar_survey': 'lsst', 'mstar_band': 'z03',
           'outbase': f'check_{v[0]}_{v[1]}', 'outpath': out_dir,
           'calib_color_nodesizes': [0.05] * (nb - 1), 'calib_slope_nodesizes': [0.1] * (nb - 1),
           'calib_color_maxnodes': [-1] * (nb - 1), 'calib_covmat_maxnodes': [-1] * (nb - 1)}
    if nb == 4:
        cfg.update({'calib_colormem_colormodes': [0, 1, 2], 'calib_colormem_zbounds': [0.35, 0.72], 'calib_colormem_sigint': [0.05, 0.03, 0.03]})
    else:
        cfg.update({'calib_colormem_colormodes': [0, 1], 'calib_colormem_zbounds': [0.72], 'calib_colormem_sigint': [0.03, 0.03]})

    # the nside 8 pixel (ring) with the most galaxies
    master = fitsio.read(cfg['galfile'])
    p = hp.ang2pix(8, master['ra_pix'][0], master['dec_pix'][0], lonlat=True)
    cfg.update({'nside': 8, 'hpix': [int(np.bincount(p, weights=master['ngals'][0]).argmax())], 'border': 0.0})

    path = os.path.join(out_dir, cfg['outbase'] + '.yml')
    yaml.safe_dump(cfg, open(path, 'w'))
    return path

In [ ]:
rm_ok = {}
for v in files:
    print(f'\n{v[0]} {v[1]}')
    config = redmapper.Configuration(make_config(v))
    print(f'  config ok: area {config.area:.1f}, bands {config.bands}, zeropoint {config.zeropoint}, '
          f'limmag_catalog {config.limmag_catalog}, hpix {config.d.hpix} (nside {config.d.nside})')

    gals = redmapper.GalaxyCatalog.from_galfile(config.galfile, nside=config.d.nside, hpix=config.d.hpix, border=config.border)
    mask = redmapper.mask.HPMask(config, include_maskgals=False)
    depth = redmapper.DepthMap(config)
    lim, exptime, m50 = depth.get_depth_values(gals.ra, gals.dec)
    spec = redmapper.GalaxyCatalog.from_fits_file(config.specfile)
    print(f'  {len(gals)} galaxies, refmag {gals.refmag.min():.2f}-{gals.refmag.max():.2f}; '
          f'{mask.compute_radmask(gals.ra, gals.dec).mean():.3f} pass the mask; '
          f'depth for {np.isfinite(lim).mean() * 100:.1f}%; {len(spec)} spec-z')
    rm_ok[v] = np.isfinite(lim).all() and len(gals) > 0 and len(spec) > 0

## Summary